# Step 1: Clean LeetCode JSONL

**What this notebook does (in order):**
1. Loads `leetcode_problems_enriched.jsonl` (source of truth — not the CSV)
2. Deduplicates on `questionFrontendId` (keeps latest `fetched_at`)
3. Drops 8 bloated/useless fields (`codeSnippets`, `metaData`, etc.)
4. Renames & type-casts columns
5. Normalizes `topicTags` → list of slugs
6. Flattens `stats` → `acRate`, `totalAccepted`, `totalSubmission`
7. Normalizes `similarQuestions` → list of slugs
8. Merges paid problem content from `data/leetcode/` (doocs/leetcode repo)
9. Writes clean `data/my/leetcode_clean.jsonl`

In [1]:
# Cell 1 — Imports & Paths
import json
import re
import pandas as pd
from pathlib import Path

# All paths via pathlib — no hardcoded string slashes
PROJECT_ROOT = Path(r"C:/PRAKSHIT/VS CODE/Prep Assistant Project/placed_in")
JSONL_IN     = PROJECT_ROOT / "data" / "my" / "leetcode_problems_enriched.jsonl"
LEETCODE_DIR = PROJECT_ROOT / "data" / "leetcode" / "solution"
JSONL_OUT    = PROJECT_ROOT / "data" / "my" / "leetcode_clean.jsonl"
ERROR_LOG    = PROJECT_ROOT / "data" / "my" / "step1_errors.log"

print(f"Source JSONL   : {JSONL_IN}")
print(f"Source exists  : {JSONL_IN.exists()}")
print(f"LeetCode repo  : {LEETCODE_DIR}")
print(f"Repo exists    : {LEETCODE_DIR.exists()}")
print(f"Output         : {JSONL_OUT}")

Source JSONL   : C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\my\leetcode_problems_enriched.jsonl
Source exists  : True
LeetCode repo  : C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\leetcode\solution
Repo exists    : True
Output         : C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\my\leetcode_clean.jsonl


In [2]:
# Cell 2 — Load JSONL into DataFrame
records = []
with JSONL_IN.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df_raw = pd.DataFrame(records)
print(f"Total raw records    : {len(df_raw):,}")
print(f"Total columns        : {len(df_raw.columns)}")
print(f"Columns: {list(df_raw.columns)}")

Total raw records    : 4,292
Total columns        : 22
Columns: ['source_slug', 'fetch_failed', 'fetched_at', 'questionId', 'questionFrontendId', 'title', 'titleSlug', 'content', 'difficulty', 'likes', 'dislikes', 'isPaidOnly', 'hints', 'exampleTestcases', 'topicTags', 'codeSnippets', 'stats', 'similarQuestions', 'metaData', 'solution', 'companyTagStats', 'categoryTitle']


In [3]:
# Cell 3 — Deduplicate on questionFrontendId (keep latest fetched_at)
df_raw["fetched_at"] = pd.to_datetime(df_raw["fetched_at"], errors="coerce")
df_sorted = df_raw.sort_values("fetched_at", ascending=False)
df = df_sorted.drop_duplicates(subset="questionFrontendId", keep="first").copy()
df = df.sort_values("questionFrontendId").reset_index(drop=True)

print(f"Before dedup : {len(df_raw):,} records")
print(f"After dedup  : {len(df):,} unique problems")
print(f"Duplicates   : {len(df_raw) - len(df):,} removed")

Before dedup : 4,292 records
After dedup  : 3,943 unique problems
Duplicates   : 349 removed


In [4]:
# Cell 4 — Drop Unwanted Fields
# Note: In the JSONL, 'solution' IS a nested object {id, canSeeDetail, paidOnly}
# (unlike the CSV where it's split into solutionId, solutionCanSeeDetail, solutionPaidOnly)
COLS_TO_DROP = [
    "codeSnippets",      # 15+ language stubs per problem (~80% of file size)
    "metaData",          # function signature JSON schema
    "companyTagStats",   # always null in the entire dataset
    "solution",          # nested {id, canSeeDetail, paidOnly} — not needed
    "source_slug",       # exact duplicate of titleSlug
    "fetched_at",        # scraping operational metadata
    "fetch_failed",      # scraping operational metadata
    "solutionHasVideo",  # not relevant for prep assistant
]

existing_drops = [c for c in COLS_TO_DROP if c in df.columns]
missing_drops  = [c for c in COLS_TO_DROP if c not in df.columns]

df.drop(columns=existing_drops, inplace=True)

print(f"Dropped ({len(existing_drops)}): {existing_drops}")
if missing_drops:
    print(f"Already absent  : {missing_drops}")
print(f"Remaining columns ({len(df.columns)}): {list(df.columns)}")

Dropped (7): ['codeSnippets', 'metaData', 'companyTagStats', 'solution', 'source_slug', 'fetched_at', 'fetch_failed']
Already absent  : ['solutionHasVideo']
Remaining columns (15): ['questionId', 'questionFrontendId', 'title', 'titleSlug', 'content', 'difficulty', 'likes', 'dislikes', 'isPaidOnly', 'hints', 'exampleTestcases', 'topicTags', 'stats', 'similarQuestions', 'categoryTitle']


In [5]:
# Cell 5 — Rename & Type-cast Columns
df.rename(columns={
    "questionFrontendId" : "id",
    "questionId"         : "internalId",
    "titleSlug"          : "slug",
    "categoryTitle"      : "category",
}, inplace=True)

df["id"]         = pd.to_numeric(df["id"], errors="coerce").astype("Int64")
df["internalId"] = pd.to_numeric(df["internalId"], errors="coerce").astype("Int64")
df["isPaidOnly"] = df["isPaidOnly"].astype(bool)
df["likes"]      = pd.to_numeric(df.get("likes"), errors="coerce").astype("Int64")
df["dislikes"]   = pd.to_numeric(df.get("dislikes"), errors="coerce").astype("Int64")

print(df[["id", "title", "slug", "difficulty", "isPaidOnly", "likes"]].head(5).to_string())

     id                         title                          slug difficulty  isPaidOnly  likes
0     1                       Two Sum                       two-sum       Easy       False  68909
1    10   Regular Expression Matching   regular-expression-matching       Hard       False  13363
2   100                     Same Tree                     same-tree       Easy       False  12942
3  1000  Minimum Cost to Merge Stones  minimum-cost-to-merge-stones       Hard       False   2643
4  1001             Grid Illumination             grid-illumination       Hard       False    657


In [6]:
# Cell 6 — Normalize topicTags → list of slugs
# JSONL source: [{"name": "Array", "slug": "array"}, ...]
def parse_topic_tags(val):
    if val is None or (isinstance(val, float)):
        return []
    if isinstance(val, list):
        return [t.get("slug", "") for t in val if isinstance(t, dict) and t.get("slug")]
    if isinstance(val, str):
        # Fallback: pipe-separated topic names (e.g. "Array|Hash Table")
        return [s.strip().lower().replace(" ", "-") for s in val.split("|") if s.strip()]
    return []

df["topicTags"] = df["topicTags"].apply(parse_topic_tags)

# Verify
print("Sample topicTags:")
for _, row in df.head(3).iterrows():
    print(f"  #{row['id']} {row['slug']}: {row['topicTags']}")

Sample topicTags:
  #1 two-sum: ['array', 'hash-table']
  #10 regular-expression-matching: ['string', 'dynamic-programming', 'recursion']
  #100 same-tree: ['tree', 'depth-first-search', 'breadth-first-search', 'binary-tree']


In [7]:
# Cell 7 — Flatten stats → acRate, totalAccepted, totalSubmission
def parse_stats(val):
    empty = {"acRate": None, "totalAccepted": None, "totalSubmission": None}
    if val is None or (isinstance(val, float)):
        return empty
    try:
        s = json.loads(val) if isinstance(val, str) else val
        return {
            "acRate"         : s.get("acRate"),
            "totalAccepted"  : s.get("totalAcceptedRaw"),
            "totalSubmission": s.get("totalSubmissionRaw"),
        }
    except Exception:
        return empty

stats_expanded = df["stats"].apply(parse_stats).apply(pd.Series)
df = pd.concat([df.drop(columns=["stats"]), stats_expanded], axis=1)

print(df[["slug", "acRate", "totalAccepted", "totalSubmission"]].head(5).to_string())

                           slug acRate  totalAccepted  totalSubmission
0                       two-sum  57.5%       21838925         37973045
1   regular-expression-matching  31.0%        1400507          4522627
2                     same-tree  67.2%        3443160          5127504
3  minimum-cost-to-merge-stones  46.1%          52755           114319
4             grid-illumination  39.3%          27485            69909


In [8]:
# Cell 8 — Normalize similarQuestions → flat list of title slugs
# JSONL: JSON string "[{\"title\": \"3Sum\", \"titleSlug\": \"3sum\", ...}, ...]"
def parse_similar(val):
    if val is None or (isinstance(val, float)):
        return []
    try:
        items = json.loads(val) if isinstance(val, str) else val
        if isinstance(items, list):
            return [i.get("titleSlug", "") for i in items if isinstance(i, dict) and i.get("titleSlug")]
    except Exception:
        pass
    return []

df["similarQuestions"] = df["similarQuestions"].apply(parse_similar)

print("Sample similarQuestions:")
for _, row in df.head(3).iterrows():
    print(f"  #{row['id']} {row['slug']}: {row['similarQuestions']}")

Sample similarQuestions:
  #1 two-sum: ['3sum', '4sum', 'two-sum-ii-input-array-is-sorted', 'two-sum-iii-data-structure-design', 'subarray-sum-equals-k', 'two-sum-iv-input-is-a-bst', 'two-sum-less-than-k', 'max-number-of-k-sum-pairs', 'count-good-meals', 'count-number-of-pairs-with-absolute-difference-k', 'number-of-pairs-of-strings-with-concatenation-equal-to-target', 'find-all-k-distant-indices-in-an-array', 'first-letter-to-appear-twice', 'number-of-excellent-pairs', 'number-of-arithmetic-triplets', 'node-with-highest-edge-score', 'check-distances-between-same-letters', 'find-subarrays-with-equal-sum', 'largest-positive-integer-that-exists-with-its-negative', 'number-of-distinct-averages', 'count-pairs-whose-sum-is-less-than-target']
  #10 regular-expression-matching: ['wildcard-matching']
  #100 same-tree: []


In [ ]:
# Cell 9 — Helper Functions for Paid Problem Content Merge
def find_paid_readme(problem_id: int) -> Path | None:
    """
    Locate README_EN.md for a paid problem in the doocs/leetcode repo.
    Folder structure: solution/XXXX-XXXX/XXXX.Title Name/README_EN.md
    """
    padded = f"{problem_id:04d}"
    lo     = (problem_id // 100) * 100
    hi     = lo + 99
    range_dir = LEETCODE_DIR / f"{lo:04d}-{hi:04d}"

    if not range_dir.exists():
        return None

    # Scan for folder whose name STARTS WITH the padded problem ID
    for folder in range_dir.iterdir():
        if folder.is_dir() and folder.name.startswith(padded):
            readme = folder / "README_EN.md"
            if readme.exists():
                return readme
    return None


def extract_description(readme_path: Path) -> str:
    """Extract the English problem description block from README_EN.md"""
    content = readme_path.read_text(encoding="utf-8")
    match = re.search(
        r"<!-- description:start -->(.*?)<!-- description:end -->",
        content,
        re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    # Fallback: return full README if no description block found
    return content.strip()


# Quick test — problem #265 (Paint House II, paid)
test_readme = find_paid_readme(265)
print(f"Test #265 README path : {test_readme}")
if test_readme:
    desc = extract_description(test_readme)
    print(f"Description (first 300 chars):\n{desc[:300]}")
else:
    print("README not found — check LEETCODE_DIR path")

Test #265 README path : C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\leetcode\solution\0200-0299\0265.Paint House II\README_EN.md
Description (first 300 chars):
<p>There are a row of <code>n</code> houses, each house can be painted with one of the <code>k</code> colors. The cost of painting each house with a certain color is different. You have to paint all the houses such that no two adjacent houses have the same color.</p>

<p>The cost of painting each ho


In [10]:
# Cell 10 — Merge Paid Problem Content
# For every problem where isPaidOnly=True and content is empty,
# pull content from the doocs/leetcode README_EN.md

errors = []
content_source = []

for idx, row in df.iterrows():
    is_paid = bool(row.get("isPaidOnly", False))
    current_content = str(row.get("content", "") or "").strip()

    if is_paid and not current_content:
        pid = int(row["id"])
        readme = find_paid_readme(pid)
        if readme:
            merged = extract_description(readme)
            df.at[idx, "content"] = merged
            content_source.append("leetcode_repo")
        else:
            errors.append(f"#{pid} {row['title']}: README not found")
            content_source.append("missing")
    else:
        content_source.append("leetcode_api")

df["contentSource"] = content_source

# Save error log
ERROR_LOG.write_text("\n".join(errors), encoding="utf-8")

source_counts = pd.Series(content_source).value_counts()
print("Content source breakdown:")
for src, cnt in source_counts.items():
    print(f"  {src:<20}: {cnt:,}")
print(f"\nProblems with missing content (logged): {len(errors)}")
if errors:
    print(f"  First 5: {errors[:5]}")

Content source breakdown:
  leetcode_api        : 3,175
  leetcode_repo       : 768

Problems with missing content (logged): 0


In [11]:
# Cell 11 — Select Final Columns & Write Output JSONL
FINAL_COLS = [
    "id", "internalId", "title", "slug", "difficulty", "category",
    "isPaidOnly", "contentSource", "content",
    "topicTags", "hints", "exampleTestcases", "similarQuestions",
    "likes", "dislikes", "acRate", "totalAccepted", "totalSubmission",
]

# Only include columns that actually exist in the DataFrame
final_cols = [c for c in FINAL_COLS if c in df.columns]
missing_cols = [c for c in FINAL_COLS if c not in df.columns]
if missing_cols:
    print(f"Note — these planned columns were not found: {missing_cols}")

df_out = df[final_cols].copy()

# Write JSONL — one JSON object per line
with JSONL_OUT.open("w", encoding="utf-8") as f:
    for _, row in df_out.iterrows():
        record = row.to_dict()
        # Convert pandas NA/NaT to None for clean JSON
        record = {k: (None if pd.isna(v) else v) for k, v in record.items()
                  if not isinstance(v, list)}
        # Preserve list fields as-is
        for col in ["topicTags", "hints", "similarQuestions"]:
            if col in row and isinstance(row[col], list):
                record[col] = row[col]
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

file_mb = JSONL_OUT.stat().st_size / 1024 / 1024
print(f"✅ Wrote {len(df_out):,} records to:")
print(f"   {JSONL_OUT}")
print(f"   File size: {file_mb:.1f} MB")

✅ Wrote 3,943 records to:
   C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\my\leetcode_clean.jsonl
   File size: 11.7 MB


In [12]:
# Cell 12 — Sanity Checks
print("=" * 55)
print("SANITY CHECKS")
print("=" * 55)

# Check #265 Paint House II (should have content from leetcode_repo)
row_265 = df_out[df_out["id"] == 265]
if not row_265.empty:
    r = row_265.iloc[0]
    print(f"\n#265 Paint House II:")
    print(f"  isPaidOnly    : {r['isPaidOnly']}")
    print(f"  contentSource : {r['contentSource']}")
    print(f"  content len   : {len(str(r['content']))} chars")
    print(f"  content[:100] : {str(r['content'])[:100]}")
else:
    print("#265 not found in output")

# Check #1 Two Sum (should have content from leetcode_api)
row_1 = df_out[df_out["id"] == 1]
if not row_1.empty:
    r = row_1.iloc[0]
    print(f"\n#1 Two Sum:")
    print(f"  contentSource : {r['contentSource']}")
    print(f"  topicTags     : {r['topicTags']}")

# Distributions
print(f"\nDifficulty distribution:")
print(df_out["difficulty"].value_counts().to_string())

print(f"\nCategory distribution:")
print(df_out["category"].value_counts().to_string())

print(f"\nContent source distribution:")
print(df_out["contentSource"].value_counts().to_string())

# Empty content check
empty = df_out[
    df_out["content"].isna() |
    (df_out["content"].astype(str).str.strip() == "") |
    (df_out["content"].astype(str).str.strip() == "None")
]
print(f"\nProblems with empty/None content: {len(empty)}")
if not empty.empty:
    print(empty[["id", "title", "isPaidOnly", "contentSource"]].head(10).to_string())

SANITY CHECKS

#265 Paint House II:
  isPaidOnly    : True
  contentSource : leetcode_repo
  content len   : 1569 chars
  content[:100] : <p>There are a row of <code>n</code> houses, each house can be painted with one of the <code>k</code

#1 Two Sum:
  contentSource : leetcode_api
  topicTags     : ['array', 'hash-table']

Difficulty distribution:
difficulty
Medium    2061
Easy       946
Hard       936

Category distribution:
category
Algorithms     3525
Database        323
JavaScript       67
pandas           15
Concurrency       9
Shell             4

Content source distribution:
contentSource
leetcode_api     3175
leetcode_repo     768

Problems with empty/None content: 0
